## Baseline Model Results

Using segmented audio (5-second windows) significantly improved performance:

- Feature samples increased from ~800 to ~4,800
- Validation accuracy improved from ~0.50 to ~0.78
- Model performance is now consistent across all classes

This demonstrates the importance of aligning data preprocessing with the natural structure of bird vocalizations.

In [1]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

In [2]:
DATA_DIR = ".."

TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
TAXONOMY_CSV = os.path.join(DATA_DIR, "taxonomy.csv")
SAMPLE_SUB_CSV = os.path.join(DATA_DIR, "sample_submission.csv")
TEST_DIR = os.path.join(DATA_DIR, "test_soundscapes")
TRAIN_AUDIO_DIR = os.path.join(DATA_DIR, "train_audio")

In [3]:
print("TEST_DIR exists:", os.path.exists(TEST_DIR))

if os.path.exists(TEST_DIR):
    test_files = sorted([f for f in os.listdir(TEST_DIR) if f.endswith(".ogg")])
    print("Number of test files:", len(test_files))
    print("First few test files:", test_files[:5])
else:
    print("TEST_DIR does not exist")

TEST_DIR exists: True
Number of test files: 0
First few test files: []


In [4]:
print("Current working directory:", os.getcwd())
print("TRAIN_CSV exists:", os.path.exists(TRAIN_CSV), TRAIN_CSV)
print("TAXONOMY_CSV exists:", os.path.exists(TAXONOMY_CSV), TAXONOMY_CSV)
print("SAMPLE_SUB_CSV exists:", os.path.exists(SAMPLE_SUB_CSV), SAMPLE_SUB_CSV)
print("TEST_DIR exists:", os.path.exists(TEST_DIR), TEST_DIR)
print("TRAIN_AUDIO_DIR exists:", os.path.exists(TRAIN_AUDIO_DIR), TRAIN_AUDIO_DIR)

Current working directory: c:\Users\rcolo\OneDrive\Desktop\BirdCLEF\notebooks
TRAIN_CSV exists: True ..\train.csv
TAXONOMY_CSV exists: True ..\taxonomy.csv
SAMPLE_SUB_CSV exists: True ..\sample_submission.csv
TEST_DIR exists: True ..\test_soundscapes
TRAIN_AUDIO_DIR exists: True ..\train_audio


In [5]:
train = pd.read_csv(TRAIN_CSV)
taxonomy = pd.read_csv(TAXONOMY_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB_CSV)
row_id = sample_sub.loc[0, "row_id"]
soundscape_id, end_time = row_id.rsplit("_", 1)
audio_file = f"{soundscape_id}.ogg"
audio_path = os.path.join(TEST_DIR, audio_file)

print("row_id:", row_id)
print("audio_file:", audio_file)
print("audio exists:", os.path.exists(audio_path))

row_id: BC2026_Test_0001_S05_20250227_010002_5
audio_file: BC2026_Test_0001_S05_20250227_010002.ogg
audio exists: False


In [6]:
top_labels = train["primary_label"].value_counts().head(10).index.tolist()

subset = (
    train[train["primary_label"].isin(top_labels)]
    .dropna(subset=["primary_label", "filename", "collection"])
    .sample(800, random_state=42)
    .reset_index(drop=True)
)

print("Top labels:", top_labels)
subset["primary_label"].value_counts()

Top labels: ['rubthr1', 'banana', 'fepowl', 'soulap1', 'houspa', 'coffal1', 'osprey', 'socfly1', 'compau', 'yeofly1']


primary_label
banana     91
socfly1    88
yeofly1    88
fepowl     84
coffal1    83
compau     82
osprey     74
rubthr1    73
houspa     69
soulap1    68
Name: count, dtype: int64

In [7]:
top_labels = train["primary_label"].value_counts().head(10).index.tolist()

subset = (
    train[train["primary_label"].isin(top_labels)]
    .dropna(subset=["primary_label", "filename", "collection"])
    .sample(800, random_state=42)
    .reset_index(drop=True)
)

print("Top labels:", top_labels)
subset["primary_label"].value_counts()

Top labels: ['rubthr1', 'banana', 'fepowl', 'soulap1', 'houspa', 'coffal1', 'osprey', 'socfly1', 'compau', 'yeofly1']


primary_label
banana     91
socfly1    88
yeofly1    88
fepowl     84
coffal1    83
compau     82
osprey     74
rubthr1    73
houspa     69
soulap1    68
Name: count, dtype: int64

In [8]:
def get_train_audio_path(row):
    filename = row["filename"]
    collection = row["collection"]

    option_1 = os.path.join(TRAIN_AUDIO_DIR, filename)
    option_2 = os.path.join(TRAIN_AUDIO_DIR, collection, filename)

    if os.path.exists(option_1):
        return option_1
    if os.path.exists(option_2):
        return option_2

    return None

In [9]:
def extract_segment_features(y, sr):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20)
    delta_mfcc = librosa.feature.delta(mfcc)
    delta2_mfcc = librosa.feature.delta(mfcc, order=2)

    zcr = librosa.feature.zero_crossing_rate(y)
    spec_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spec_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)

    features = np.concatenate([
        mfcc.mean(axis=1),
        mfcc.std(axis=1),
        delta_mfcc.mean(axis=1),
        delta_mfcc.std(axis=1),
        delta2_mfcc.mean(axis=1),
        delta2_mfcc.std(axis=1),
        [zcr.mean(), zcr.std()],
        [spec_centroid.mean(), spec_centroid.std()],
        [spec_bandwidth.mean(), spec_bandwidth.std()]
    ])

    return features

In [10]:
SEGMENT_SECONDS = 5
SR = 32000

X = []
y = []

for _, row in subset.iterrows():
    path = get_train_audio_path(row)
    if path is None:
        continue

    try:
        audio, sr = librosa.load(path, sr=SR)
    except Exception:
        continue

    segment_len = SR * SEGMENT_SECONDS

    for start in range(0, len(audio), segment_len):
        segment = audio[start:start + segment_len]
        if len(segment) < segment_len:
            continue

        features = extract_segment_features(segment, sr)
        X.append(features)
        y.append(row["primary_label"])

X = np.array(X)
y = np.array(y)

print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)
print("Classes:", len(np.unique(y)))

Feature matrix shape: (4895, 126)
Target shape: (4895,)
Classes: 10


In [11]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y_encoded)

trained_labels = label_encoder.classes_.tolist()
print("Trained labels:", trained_labels)

Trained labels: ['banana', 'coffal1', 'compau', 'fepowl', 'houspa', 'osprey', 'rubthr1', 'socfly1', 'soulap1', 'yeofly1']


In [12]:
def predict_window_probs(segment, sr, model, label_encoder, submission_species):
    features = extract_segment_features(segment, sr).reshape(1, -1)
    probs = model.predict_proba(features)[0]

    class_probs = dict(zip(label_encoder.classes_, probs))

    row = {species: 0.0 for species in submission_species}
    for species, prob in class_probs.items():
        if species in row:
            row[species] = float(prob)

    return row

In [13]:
test_files = sorted([
    f for f in os.listdir(TEST_DIR)
    if f.endswith(".ogg")
])

print("Number of test soundscapes:", len(test_files))
print(test_files[:5])

Number of test soundscapes: 0
[]


In [14]:
submission = sample_sub.copy()
submission.head()

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


In [15]:
def parse_row_id(row_id):
    soundscape_id, end_time = row_id.rsplit("_", 1)
    return soundscape_id, int(end_time)

In [16]:
soundscape_to_rows = {}

for i, row in submission.iterrows():
    row_id = row["row_id"]
    soundscape_id, end_time = parse_row_id(row_id)

    if soundscape_id not in soundscape_to_rows:
        soundscape_to_rows[soundscape_id] = []

    soundscape_to_rows[soundscape_id].append((i, end_time))

print("Unique soundscapes in submission:", len(soundscape_to_rows))
print("Example soundscape:", next(iter(soundscape_to_rows)) if soundscape_to_rows else "None")

Unique soundscapes in submission: 1
Example soundscape: BC2026_Test_0001_S05_20250227_010002


In [17]:
soundscape_to_rows = {}

for i, row in submission.iterrows():
    row_id = row["row_id"]
    soundscape_id, end_time = parse_row_id(row_id)

    if soundscape_id not in soundscape_to_rows:
        soundscape_to_rows[soundscape_id] = []

    soundscape_to_rows[soundscape_id].append((i, end_time))

updated_rows = 0
missing_files = 0
failed_loads = 0

for soundscape_id, row_info in soundscape_to_rows.items():
    audio_file = f"{soundscape_id}.ogg"
    audio_path = os.path.join(TEST_DIR, audio_file)

    if not os.path.exists(audio_path):
        missing_files += len(row_info)
        continue

    try:
        audio, sr = librosa.load(audio_path, sr=SR)
    except Exception:
        failed_loads += len(row_info)
        continue

    required_len = SEGMENT_SECONDS * SR

    for row_index, end_time in row_info:
        start_sample = max(0, (end_time - SEGMENT_SECONDS) * SR)
        end_sample = end_time * SR
        segment = audio[start_sample:end_sample]

        if len(segment) < required_len:
            padded = np.zeros(required_len, dtype=np.float32)
            padded[:len(segment)] = segment
            segment = padded

        pred_dict = predict_window_probs(
            segment=segment,
            sr=sr,
            model=model,
            label_encoder=label_encoder,
            submission_species=submission_species
        )

        for species, prob in pred_dict.items():
            submission.at[row_index, species] = prob

        updated_rows += 1

print("Updated rows:", updated_rows)
print("Missing file rows:", missing_files)
print("Failed load rows:", failed_loads)

Updated rows: 0
Missing file rows: 3
Failed load rows: 0


In [18]:
submission.to_csv("submission.csv", index=False)
submission.head()

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


In [19]:
submission.head(10)

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,...,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274


In [20]:
submission.iloc[:5, :8]

,row_id,1161364,116570,1176823,1491113,1595929,209233,22930
0,BC2026_Test_0001_S05_20250227_010002_5,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
1,BC2026_Test_0001_S05_20250227_010002_10,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
2,BC2026_Test_0001_S05_20250227_010002_15,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274,0.004274
